# Flight Delay Prediction: Data Cleaning (standalone)

Independent data-cleaning pass for the [Zindi Flight Delay Prediction Challenge](https://zindi.africa/competitions/flight-delay-prediction-challenge) (Tunisair), built on its own branch (not merged into `main`) as a separate take from the team's existing `data_cleaning.ipynb` / `04_flight_delay_modeling.ipynb`.

Loads `data/Train.csv`, fixes the date-parsing quirks, and writes the result to `data/cleaned_data_standalone.csv` for [[02_feature_engineering_and_modeling.ipynb]] to pick up -- mirroring the same clean-then-model handoff pattern already used elsewhere in this repo, just under a separate filename so it doesn't overwrite `data/cleaned_data.csv`.

## Load Data

In [1]:
import pandas as pd

train = pd.read_csv("data/Train.csv")
train.head()

,ID,DATOP,FLTID,DEPSTN,ARRSTN,STD,STA,STATUS,AC,target
0,train_id_0,2016-01-03,TU 0712,CMN,TUN,2016-01-03 10:30:00,2016-01-03 12.55.00,ATA,TU 32AIMN,260.0
1,train_id_1,2016-01-13,TU 0757,MXP,TUN,2016-01-13 15:05:00,2016-01-13 16.55.00,ATA,TU 31BIMO,20.0
2,train_id_2,2016-01-16,TU 0214,TUN,IST,2016-01-16 04:10:00,2016-01-16 06.45.00,ATA,TU 32AIMN,0.0
3,train_id_3,2016-01-17,TU 0480,DJE,NTE,2016-01-17 14:10:00,2016-01-17 17.00.00,ATA,TU 736IOK,0.0
4,train_id_4,2016-01-17,TU 0338,TUN,ALG,2016-01-17 14:30:00,2016-01-17 15.50.00,ATA,TU 320IMU,22.0


## Fix Date Parsing

Two issues with the raw date columns:

1. `STA` uses dots as the time separator (`2016-01-03 12.55.00`) instead of colons like `STD`/`DATOP` (`2016-01-03 10:30:00`), which pandas can't parse at all -- every `STA` value comes out `NaT` if left as-is. Normalising dots to colons first fixes this.
2. Parsing with `errors="coerce"` (rather than relying on `read_csv`'s `parse_dates`) keeps every column a proper `datetime64` dtype even when a handful of values are malformed, instead of silently leaving the whole column as `object` dtype.

In [2]:
train["STA"] = train["STA"].str.replace(".", ":", regex=False)
for col in ["DATOP", "STD", "STA"]:
    train[col] = pd.to_datetime(train[col], errors="coerce")

train.info()

<class 'pandas.DataFrame'>
RangeIndex: 107833 entries, 0 to 107832
Data columns (total 10 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   ID      107833 non-null  str           
 1   DATOP   107833 non-null  datetime64[us]
 2   FLTID   107833 non-null  str           
 3   DEPSTN  107833 non-null  str           
 4   ARRSTN  107833 non-null  str           
 5   STD     107833 non-null  datetime64[us]
 6   STA     107833 non-null  datetime64[us]
 7   STATUS  107833 non-null  str           
 8   AC      107833 non-null  str           
 9   target  107833 non-null  float64       
dtypes: datetime64[us](3), float64(1), str(6)
memory usage: 8.2 MB


## Save Cleaned Data

In [3]:
train.to_csv("data/cleaned_data_standalone.csv", index=False)
print(f"Saved {len(train)} rows to data/cleaned_data_standalone.csv")

Saved 107833 rows to data/cleaned_data_standalone.csv
